# ScienceQA Visual Challenge: Starter Notebook

This notebook provides a starting point for the ScienceQA Visual Multiple-Choice Challenge. It is based on the provided baseline solution, but has been adapted to be more of a general-purpose starter.

**Objective:** Build a model that can answer visual multiple-choice questions based on scientific diagrams and text.

**Baseline Model:** `HuggingFaceTB/SmolVLM-500M-Instruct` (~500 M params)
**Fine-Tuning:** QLoRA (4-bit NF4)
**Scoring:** Multiple-choice log-likelihood

---

test

In [ ]:
# ── 0a. Mount Google Drive & set up local data ──────────────────────────────
from google.colab import drive
from pathlib import Path
import os, shutil

drive.mount("/content/drive")

# Edit this to match the folder you created in Google Drive
DRIVE_DATA = Path("/content/drive/MyDrive/DLproj3")
LOCAL_DATA = Path("/content/data")
LOCAL_DATA.mkdir(exist_ok=True)

# Copy CSVs from Drive to local storage
for csv_name in ["train.csv", "val.csv", "test.csv"]:
    dst = LOCAL_DATA / csv_name
    if not dst.exists():
        shutil.copy(DRIVE_DATA / csv_name, dst)
        print(f"Copied {csv_name}")

# Symlink the images folder from Drive (instant — no copying)
images_link = LOCAL_DATA / "images"
if images_link.is_symlink():
    images_link.unlink()
if not images_link.exists():
    os.symlink(DRIVE_DATA / "images" / "images", images_link)
    print("Linked images from Drive.")

splits = [p.name for p in images_link.iterdir()]
print(f"Data ready. Image splits found: {splits}")


In [2]:
# ── 0. Install libraries ──────────────────────────────────────────
# Run this cell to install the necessary Python packages.
!pip install -q transformers==4.57.6 peft==0.18.1 bitsandbytes accelerate datasets pillow
!pip install peft -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.9 MB/s eta 0:00:00


In [3]:
# ── 1. Imports & Configuration ───────────────────────────────────────────────
import os
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader

# ── Reproducibility ──────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# ── Paths ────────────────────────────────────────────────────────────────────
# Adjust these paths to match your local environment
DATA_DIR   = Path("data")

# ── Model ────────────────────────────────────────────────────────────────────
MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"

# ── Basic Settings ───────────────────────────────────────────────────────────
IMG_SIZE        = 224

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Using device: cuda
GPU: Tesla T4


## 2. Load and Preprocess Data

In [5]:
# ── 2a. Load CSVs ─────────────────────────────────────────────────────────────
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df   = pd.read_csv(DATA_DIR / "val.csv")
test_df  = pd.read_csv(DATA_DIR / "test.csv")

# The 'choices' column is a JSON string, so we parse it
for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print(f"Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}")
train_df.head(2)

FileNotFoundError: [Errno 2] No such file or directory: 'data/train.csv'

In [ ]:
row = train_df.iloc[0]
img_path = DATA_DIR / row["image_path"]

print(img_path)
img = Image.open(img_path).convert("RGB")
img

In [ ]:
CHOICE_LETTERS = "ABCDEFGHIJ"

def build_prompt(row: pd.Series, include_answer: bool = False) -> str:
    context_parts = []
    lecture = row.get("lecture", "")
    hint    = row.get("hint", "")
    if pd.notna(lecture) and str(lecture).strip():
        context_parts.append(str(lecture).strip())
    if pd.notna(hint) and str(hint).strip():
        context_parts.append(str(hint).strip())
    context_str = "\n".join(context_parts)

    choices = row["choices"]
    choices_str = "\n".join(f"{CHOICE_LETTERS[i]}. {c}" for i, c in enumerate(choices))

    text = ""
    if context_str:
        text += f"Context:\n{context_str}\n\n"
    text += f"Question: {row['question']}\n"
    text += f"Choices:\n{choices_str}\n"
    text += "Answer:"

    messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": text}]}]
    prompt = processor.apply_chat_template(messages, add_generation_prompt=True)

    if include_answer:
        answer_idx = int(row["answer"])
        prompt += CHOICE_LETTERS[answer_idx]

    return prompt


In [ ]:
# ── 2c. PyTorch Dataset ───────────────────────────────────────────────────────
class ScienceQADataset(Dataset):
    def __init__(self, df: pd.DataFrame, data_dir: Path, img_size: int = 224, is_train: bool = True):
        self.df = df.reset_index(drop=True)
        self.data_dir = data_dir
        self.img_size = img_size
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.df)

    def _load_image(self, rel_path: str) -> Image.Image:
        img = Image.open(self.data_dir / rel_path).convert("RGB")
        img = img.resize((self.img_size, self.img_size), Image.BICUBIC)
        return img

    def __getitem__(self, idx: int) -> dict:
        row = self.df.iloc[idx]
        img = self._load_image(row["image_path"])

        if self.is_train:
            return {
                "image":  img,
                "text":   build_prompt(row, include_answer=True),
                "answer": int(row["answer"]),
            }
        else:
            return {
                "image":   img,
                "text":    build_prompt(row, include_answer=False),
                "choices": row["choices"],
                "answer": int(row["answer"]) if "answer" in self.df.columns else -1,
            }

train_ds = ScienceQADataset(train_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
val_ds   = ScienceQADataset(val_df,   DATA_DIR, img_size=IMG_SIZE, is_train=False)
test_ds  = ScienceQADataset(test_df,  DATA_DIR, img_size=IMG_SIZE, is_train=False)

print(f"Datasets created: train={len(train_ds)}, val={len(val_ds)}, test={len(test_ds)}")

## 3. Model Loading and Inference Example

This section loads `HuggingFaceTB/SmolVLM-500M-Instruct` and runs a quick inference example on one validation sample.

In [ ]:
# ── 3a. Load SmolVLM model in 4-bit (QLoRA) ─────────────────────────────────
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig

processor = AutoProcessor.from_pretrained(MODEL_ID)
if processor.tokenizer.pad_token is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)
model.eval()

# Quick sanity-check inference on one validation sample
sample = val_df.iloc[0]
sample_image = Image.open(DATA_DIR / sample["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
sample_prompt = build_prompt(sample, include_answer=False)

inputs = processor(
    text=[sample_prompt],
    images=[sample_image],
    return_tensors="pt",
)
inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.inference_mode():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,
    )

decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
print("Model output:", decoded)
gt = sample["answer"]
print(f"Ground-truth answer index: {gt}")


In [ ]:
def compute_logprob_continuation(model, processor, image, prompt, target_text, prompt_len=None):
    full_text = prompt + target_text  # chat template ends with newline — no extra space needed

    inputs = processor(text=[full_text], images=[image], return_tensors="pt")
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    if prompt_len is None:
        # Tokenize prompt WITH image tokens so the length is correct
        prompt_inputs = processor(text=[prompt], images=[image], return_tensors="pt")
        prompt_len = prompt_inputs["input_ids"].shape[1]

    with torch.inference_mode():
        outputs = model(**inputs)

    logits = outputs.logits[:, :-1, :]
    log_probs = torch.nn.functional.log_softmax(logits, dim=-1)

    input_ids = inputs["input_ids"][0]
    target_tokens = input_ids[prompt_len:]
    selected = log_probs[0, prompt_len - 1 : prompt_len - 1 + len(target_tokens), :]
    token_scores = selected[range(len(target_tokens)), target_tokens]

    return token_scores.sum().item()


In [ ]:
def predict_mc_choice_text(row):
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prompt = build_prompt(row, include_answer=False)

    # Compute prompt length once (with image tokens) and reuse for all choices
    prompt_inputs = processor(text=[prompt], images=[image], return_tensors="pt")
    prompt_len = prompt_inputs["input_ids"].shape[1]

    scores = []
    for i in range(len(row["choices"])):
        score = compute_logprob_continuation(
            model=model,
            processor=processor,
            image=image,
            prompt=prompt,
            target_text=CHOICE_LETTERS[i],
            prompt_len=prompt_len,
        )
        scores.append(score)

    return int(np.argmax(scores))


In [ ]:
val_preds_choice_text = []

for i, row in val_df.iterrows():
    pred = predict_mc_choice_text(row)
    val_preds_choice_text.append(pred)

    if i % 50 == 0:
        acc = np.mean(
            np.array(val_preds_choice_text)
            == val_df.iloc[:len(val_preds_choice_text)]["answer"].values
        )
        print(f"{i}/{len(val_df)} | acc: {acc:.4f}")

val_acc_choice_text = np.mean(
    np.array(val_preds_choice_text) == val_df["answer"].values
)

print("Choice-text validation accuracy:", val_acc_choice_text)

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


In [ ]:
from torch.utils.data import DataLoader

def collate_fn(batch):
    images = [b["image"] for b in batch]
    texts  = [b["text"]  for b in batch]
    inputs = processor(text=texts, images=images, return_tensors="pt", padding=True)
    labels = inputs["input_ids"].clone()
    labels[labels == processor.tokenizer.pad_token_id] = -100
    inputs["labels"] = labels
    return inputs

combined_df = pd.concat([train_df, val_df], ignore_index=True)
combined_ds = ScienceQADataset(combined_df, DATA_DIR, img_size=IMG_SIZE, is_train=True)
train_loader = DataLoader(combined_ds, batch_size=1, shuffle=True,
                          collate_fn=collate_fn, num_workers=0)

print(f"Training on {len(combined_ds)} examples")

In [ ]:
# Clear GPU cache before training
torch.cuda.empty_cache()
import gc; gc.collect()

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

NUM_EPOCHS = 3
GRAD_ACCUM_STEPS = 4

optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
num_steps = NUM_EPOCHS * len(train_loader) // GRAD_ACCUM_STEPS
scheduler = get_scheduler("cosine", optimizer,
                           num_warmup_steps=max(1, num_steps // 10),
                           num_training_steps=num_steps)

model.train()
for epoch in range(NUM_EPOCHS):
    total_loss = 0
    optimizer.zero_grad()

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(model.device) for k, v in batch.items()}

        with torch.amp.autocast("cuda"):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM_STEPS

        loss.backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        if step % 50 == 0:
            print(f"Epoch {epoch+1} | Step {step}/{len(train_loader)} | "
                  f"Loss: {loss.item()*GRAD_ACCUM_STEPS:.4f}")

    print(f"Epoch {epoch+1} complete | Avg loss: {total_loss/len(train_loader):.4f}")

print("Training done!")


In [ ]:
model.save_pretrained("lora_weights")
print("LoRA weights saved.")

In [ ]:
model.eval()
print("Model set to eval mode — ready for prediction.")

In [ ]:
test_preds = []

for i, row in test_df.iterrows():
    pred = predict_mc_choice_text(row)
    test_preds.append(pred)
    if i % 50 == 0:
        print(f"{i}/{len(test_df)}")

In [ ]:
submission = pd.DataFrame({
    "id":     test_df["id"],
    "answer": test_preds,
})
submission.to_csv("submission.csv", index=False)
print(submission["answer"].value_counts().sort_index())
print(submission.shape)
submission.head()

In [ ]:
from google.colab import files
files.download("submission.csv")

**Old Pipeline (keep for future reference - 1st submission)**

In [ ]:
# ── 4a. Helper: extract answer from generated text ──────────────────────────
def parse_answer(text, num_choices):
    text = text.strip().upper()
    for i, letter in enumerate(CHOICE_LETTERS[:num_choices]):
        if f"ANSWER: {letter}" in text or text.startswith(letter):
            return i
    for i, letter in enumerate(CHOICE_LETTERS[:num_choices]):
        if letter in text[:20]:
            return i
    return 0

In [ ]:
# ── 4b. Run validation baseline using generation ────────────────────────────
def predict_one(row):
    image = Image.open(DATA_DIR / row["image_path"]).convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    prompt = build_prompt(row, include_answer=False)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )
    inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

    with torch.inference_mode():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
        )

    decoded = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    return parse_answer(decoded, int(row["num_choices"]))

In [ ]:
# ── 4c. Evaluate on validation set ──────────────────────────────────────────
val_preds = []

for i, row in val_df.iterrows():
    pred = predict_one(row)
    val_preds.append(pred)

    if i % 50 == 0:
        acc_so_far = np.mean(np.array(val_preds) == val_df.iloc[:len(val_preds)]["answer"].values)
        print(f"{i}/{len(val_df)} | acc so far: {acc_so_far:.4f}")

val_acc = np.mean(np.array(val_preds) == val_df["answer"].values)
print("Validation accuracy:", val_acc)

In [ ]:
# Save validation predictions
val_results = val_df[["id", "answer"]].copy()
val_results["pred"] = val_preds
val_results["correct"] = val_results["answer"] == val_results["pred"]

val_results.to_csv("val_generation_baseline.csv", index=False)

print("Validation accuracy:", val_results["correct"].mean())
val_results.head()

In [ ]:
test_preds = []

for i, row in test_df.iterrows():
    pred = predict_one(row)
    test_preds.append(pred)

    if i % 50 == 0:
        print(f"{i}/{len(test_df)}")

In [ ]:
submission = pd.DataFrame({
    "id": test_df["id"],
    "answer": test_preds,
})

submission.to_csv("submission.csv", index=False)
submission.head()

In [ ]:
# ── 5c. Sanity checks ───────────────────────────────────────────────────────
print("Shape:", submission.shape)
print("Columns:", submission.columns.tolist())
print("Missing answers:", submission["answer"].isna().sum())
print("Answer counts:")
print(submission["answer"].value_counts().sort_index())

assert list(submission.columns) == ["id", "answer"]
assert len(submission) == len(test_df)
assert submission["answer"].notna().all()
assert submission["answer"].apply(lambda x: isinstance(x, (int, np.integer))).all()

print("submission.csv is ready.")

In [ ]:
# ── 5d. Download submission ─────────────────────────────────────────────────
from google.colab import files
files.download("submission.csv")